# PML · Lecture 10 — Logistic Regression, end to end

Logistic regression is the **discriminative** classifier: instead of modelling how the data was generated (Lecture 9), it models the label probability directly, $p(y{=}1\mid x)=\sigma(w^\top x)$, and fits $w$ by maximum likelihood. This notebook builds it from the sigmoid up, one small visual step at a time:

1. the **sigmoid** and its clean derivative,
2. **binary cross-entropy** = the negative log-likelihood (and why not squared error),
3. the **gradient** $X^\top(\sigma(Xw)-y)$ — checked against a numerical gradient,
4. **train by gradient descent** and watch the loss fall,
5. the **decision boundary** is a line ($w^\top x=0$) + a probability heatmap,
6. **one GD step by hand**, reproduced in code,
7. **MAP = L2**: why unregularized ML diverges on separable data, and how the prior fixes it,
8. a check against **scikit-learn**, and a peek at **multiclass softmax**.

Pure `numpy` + `matplotlib` (+ `scikit-learn` for the comparison) — all in Colab. Run top to bottom.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (6.2, 4.6)
C0, C1 = '#1f77b4', '#d62728'

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

## 1. The sigmoid — squash a score into a probability

$\sigma(z)=\dfrac{1}{1+e^{-z}}$ maps any real score $z=w^\top x$ into $(0,1)$. It passes through $\sigma(0)=0.5$ (the decision threshold) and saturates toward 0 and 1. Its derivative is beautifully simple — $\sigma'(z)=\sigma(z)\,(1-\sigma(z))$ — and that identity is exactly what makes the gradient clean later.

In [ ]:
z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z), label=r'$\sigma(z)$')
plt.plot(z, sigmoid(z)*(1-sigmoid(z)), '--', label=r"$\sigma'(z)=\sigma(1-\sigma)$")
plt.axhline(0.5, color='k', lw=.5); plt.axvline(0, color='k', lw=.5)
plt.xlabel('score z = wᵀx'); plt.legend(); plt.title('The logistic (sigmoid) function'); plt.show()

# verify the derivative identity against a numerical derivative
num = (sigmoid(z+1e-5) - sigmoid(z-1e-5)) / 2e-5
print('max |σ\'(z) - numerical| =', np.max(np.abs(sigmoid(z)*(1-sigmoid(z)) - num)))

### Log-odds: the model is linear after all

The sigmoid looks nonlinear, but invert it: the **log-odds** (logit) of the prediction is exactly the linear score,
$$\ln\frac{p}{1-p}=w^\top x.$$
So logistic regression is linear *in log-odds space* — which is why its decision boundary ($p=0.5\Leftrightarrow$ log-odds $=0$) is a straight line. Verify the identity numerically:

In [ ]:
zt = np.linspace(-4, 4, 9)
p = sigmoid(zt)
logit = np.log(p / (1 - p))                          # invert the sigmoid
print('max |logit(σ(z)) - z| =', np.max(np.abs(logit - zt)))   # ~0 -> log-odds is linear in z

## 2. The data — two Gaussian blobs

A binary problem in 2-D: two clouds, labels 0 and 1. We fold a **bias** into the features as a column of ones, so $x=(1,x_1,x_2)$ and the score is $w^\top x = w_0 + w_1 x_1 + w_2 x_2$.

In [ ]:
N = 200
X0 = rng.normal([-1.5, -1.5], 1.0, size=(N // 2, 2))
X1 = rng.normal([+1.5, +1.5], 1.0, size=(N // 2, 2))
Xraw = np.vstack([X0, X1])
X = np.hstack([np.ones((N, 1)), Xraw])              # bias column: x = (1, x1, x2)
y = np.r_[np.zeros(N // 2), np.ones(N // 2)]         # targets in {0, 1}

def scatter(ax=None):
    ax = ax or plt.gca()
    for k, c in [(0, C0), (1, C1)]:
        ax.scatter(*Xraw[y == k].T, s=14, c=c, alpha=.6, label=f'class {k}')
    ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.legend(loc='upper left')
    return ax
scatter(); plt.title('Two Gaussian blobs'); plt.show()
print('design matrix X:', X.shape, ' (N x [1, x1, x2])')

> **Discriminative, not generative.** Lecture 9's generative classifier modelled the *whole* story $p(x,y)=p(y)\,p(x\mid y)$ (a Gaussian per class) and used Bayes' rule to get $p(y\mid x)$. Logistic regression skips all that and models $p(y\mid x)=\sigma(w^\top x)$ **directly** — it assumes nothing about how $x$ is distributed. As a graphical model, the features $x_n$ and the weights $w$ both point into the label $y_n$; there's no model of $x$. (Adding a Gaussian prior on $w$ — the L2 term of §8 — puts a hyperparameter above $w$, turning this into the MAP/Bayesian picture.)

## 3. The loss — binary cross-entropy is the negative log-likelihood

Each label is Bernoulli with probability $\sigma_n=\sigma(w^\top x_n)$. The negative log-likelihood is
$$\mathrm{NLL}(w)=-\sum_n\big[y_n\ln\sigma_n+(1-y_n)\ln(1-\sigma_n)\big],$$
which is exactly **binary cross-entropy** — not an invented loss, just the NLL of the Bernoulli model. Per example it's near 0 when the model is right and shoots to $\infty$ when it's **confident and wrong**:

In [ ]:
p = np.linspace(1e-3, 1-1e-3, 200)
plt.plot(p, -np.log(p),   C1, label='y = 1 loss:  −log p')
plt.plot(p, -np.log(1-p), C0, label='y = 0 loss:  −log(1−p)')
plt.xlabel('predicted p = σ(wᵀx)'); plt.ylabel('per-example loss'); plt.ylim(0, 6)
plt.legend(); plt.title('Cross-entropy: cheap when right, brutal when confidently wrong'); plt.show()

def nll(w):
    s = np.clip(sigmoid(X @ w), 1e-9, 1 - 1e-9)     # clip so log never sees 0
    return -np.sum(y*np.log(s) + (1-y)*np.log(1-s))

> **Why not squared error?** You *could* put $\tfrac12(y-\sigma)^2$, but that surface is **non-convex** in $w$ and its gradient vanishes when the model is confidently wrong. Cross-entropy is **convex** and keeps a strong gradient exactly when the model is most wrong.

## 4. The gradient — $X^\top(\sigma(Xw)-y)$, checked numerically

The sigmoid identity makes the whole derivation collapse to one line: the gradient is
$$\nabla_w\,\mathrm{NLL}=\sum_n(\sigma_n-y_n)\,x_n = X^\top(\sigma(Xw)-y),$$
i.e. **$\sum$ (prediction − target) × input**. A well-fit point ($\sigma_n=y_n$) contributes nothing. Let's confirm the analytic gradient matches a finite-difference one:

In [ ]:
def grad(w):
    return X.T @ (sigmoid(X @ w) - y)               # the ML gradient

w_test = np.array([0.3, -0.5, 0.8])
g_analytic = grad(w_test)
g_numeric = np.array([(nll(w_test + e) - nll(w_test - e)) / 2e-5
                      for e in np.eye(3) * 1e-5])
print('analytic:', g_analytic.round(4))
print('numeric :', g_numeric.round(4))
print('max abs difference =', np.max(np.abs(g_analytic - g_numeric)))   # ~1e-9 -> the formula is right

## 5. Train by gradient descent

No closed form exists (the $w$ is buried inside the nonlinear $\sigma$), but the NLL is **convex**, so gradient descent walks to *the* global minimum: $w\leftarrow w-\eta\,X^\top(\sigma(Xw)-y)$.

In [ ]:
w = np.zeros(X.shape[1])
lr = 0.1
history = []
for step in range(2000):
    pgrad = grad(w)
    w -= lr * pgrad / N                             # average gradient step
    if step % 20 == 0:
        history.append(nll(w) / N)

plt.plot(np.arange(len(history)) * 20, history)
plt.xlabel('gradient-descent step'); plt.ylabel('mean NLL'); plt.title('The loss falls monotonically'); plt.show()
acc = np.mean((sigmoid(X @ w) > 0.5) == y)
print(f'weights w = {w.round(3)}   train accuracy = {acc:.3f}')

## 6. The decision boundary is a *line*

The boundary is where the score is zero, $w^\top x=0$ (so $\sigma=0.5$) — a straight line $w_0+w_1x_1+w_2x_2=0$. The sigmoid only sets *how fast* confidence grows as you move away from it. Here's the boundary plus the full probability field $\sigma(w^\top x)$:

In [ ]:
xx, yy = np.meshgrid(np.linspace(-5, 5, 300), np.linspace(-5, 5, 300))
grid = np.c_[np.ones(xx.size), xx.ravel(), yy.ravel()]
prob = sigmoid(grid @ w).reshape(xx.shape)
im = plt.contourf(xx, yy, prob, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=.75)
plt.colorbar(im, label='P(y=1 | x)')
plt.contour(xx, yy, prob, levels=[0.5], colors='k', linewidths=2)   # the boundary wᵀx = 0
scatter(plt.gca()); plt.title('Linear boundary (σ=0.5) + probability field'); plt.show()

## 7. One gradient-descent step, by hand

Reproduce the lecture's worked example. Two 1-D points (bias folded in): $x_A=(1,2),\,y_A=1$ and $x_B=(1,-2),\,y_B=0$; start at $w=(0,0)$, learning rate $\eta=0.5$. The recipe: **predict → (prediction−target)×input, summed → step**.

In [ ]:
Xh = np.array([[1., 2.], [1., -2.]]); yh = np.array([1., 0.]); wh = np.zeros(2)
sig = sigmoid(Xh @ wh)                               # both 0.5 at w=0
g = Xh.T @ (sig - yh)                                # (-0.5)(1,2) + (0.5)(1,-2) = (0,-2)
wh_new = wh - 0.5 * g
print('predictions σ =', sig, '  gradient =', g)
print('w after one step =', wh_new, '  (matches the lecture: (0, 1))')

## 8. MAP = L2-regularized logistic regression

Pure ML can **diverge**: if the classes are *linearly separable*, the likelihood is maximized by pushing $\lVert w\rVert\to\infty$ (every prediction infinitely confident). A zero-mean Gaussian **prior** $w\sim\mathcal N(0,\lambda^{-1}I)$ gives the MAP objective $\mathrm{NLL}(w)+\tfrac{\lambda}{2}\lVert w\rVert^2$ — that's **L2 regularization**, and the gradient just gains a $+\lambda w$ term. Watch $\lVert w\rVert$ on separable data: ML never settles; L2 plateaus.

In [ ]:
Xs0 = rng.normal([-3, -3], 0.4, size=(50, 2)); Xs1 = rng.normal([3, 3], 0.4, size=(50, 2))
Xs = np.hstack([np.ones((100, 1)), np.vstack([Xs0, Xs1])]); ys = np.r_[np.zeros(50), np.ones(50)]

def train(lam, steps=50000):
    w = np.zeros(3); norms = []
    for s in range(steps):
        g = Xs.T @ (sigmoid(Xs @ w) - ys) + lam * w    # + λw is the L2 term
        w -= 0.5 * g / 100
        if s % 500 == 0: norms.append(np.linalg.norm(w))
    return w, norms

_, n_ml = train(0.0); _, n_l2 = train(5.0)
xs = np.arange(len(n_ml)) * 500
plt.plot(xs, n_ml, C1, label='ML (λ=0): ‖w‖ keeps growing → ∞')
plt.plot(xs, n_l2, C0, label='L2 (λ=5): ‖w‖ plateaus')
plt.xlabel('gradient-descent step'); plt.ylabel('‖w‖'); plt.legend()
plt.title('On separable data, unregularized weights diverge'); plt.show()

The prior keeps $w$ finite, makes the problem strictly convex, and improves generalization — it falls out of MAP, not from nowhere. (A Laplace prior gives **L1** and sparse $w$.)

## 9. Check against scikit-learn, and multiclass softmax

Our from-scratch loop is exactly what a library does inside. `LogisticRegression` (L-BFGS + built-in L2) agrees on the blobs. For **K classes**, the sigmoid generalizes to the **softmax**, $p(y{=}k\mid x)=\dfrac{e^{w_k^\top x}}{\sum_j e^{w_j^\top x}}$ — one weight vector per class, still linear boundaries.

In [ ]:
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(C=1e6).fit(Xraw, y)          # large C ≈ unregularized
print('sklearn  acc =', sk.score(Xraw, y), ' coef ≈', np.r_[sk.intercept_, sk.coef_[0]].round(2))
print('from-scratch w =', w.round(2), '  (same direction)')

# multiclass: 3 blobs, softmax regions
cen = np.array([[-2, -2], [2, -2], [0, 2.5]])
Xm = np.vstack([rng.normal(c, 0.8, size=(60, 2)) for c in cen])
ym = np.r_[np.zeros(60), np.ones(60), np.full(60, 2)]
skm = LogisticRegression().fit(Xm, ym)
gx, gy = np.meshgrid(np.linspace(-5, 5, 300), np.linspace(-5, 5, 300))
pred = skm.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
plt.contourf(gx, gy, pred, levels=[-.5,.5,1.5,2.5], colors=['#9ecae1','#fdae6b','#a1d99b'], alpha=.5)
for k, c in enumerate(['#3182bd', '#e6550d', '#31a354']):
    plt.scatter(*Xm[ym == k].T, s=12, c=c)
plt.title('Multiclass softmax: one linear boundary per pair'); plt.show()

## Your turn

1. **Learning rate.** Re-run §5 with `lr = 0.01` and `lr = 1.0`. What happens to the loss curve? When does it diverge?
2. **Regularization strength.** In §8 sweep `lam` over `[0, 1, 5, 50]` and plot the final boundary for each. How does a bigger $\lambda$ change the slope/confidence?
3. **Boundary by hand.** For $w=(-3, 1, 1)$ the boundary is $x_1+x_2=3$. Set `w = np.array([-3,1,1])` and overlay that line on the probability field (§6) to confirm.
4. **Separability.** Push the two blobs in §2 far apart (centres $\pm4$) and watch training accuracy hit 1.0 — then add L2 and compare the weight norms.

## Recap
- Logistic regression models $p(y{=}1\mid x)=\sigma(w^\top x)$ — **discriminative**, a linear boundary $w^\top x=0$.
- Its loss is **binary cross-entropy** (= the Bernoulli NLL); the gradient is $X^\top(\sigma(Xw)-y)$ — *sum (prediction − target) × input*.
- **No closed form**, but convex → **gradient descent**. **MAP = L2** keeps $w$ finite on separable data.
- **Softmax** is the K-class generalization. `sklearn.linear_model.LogisticRegression` is the loop you just wrote, tuned.